# Replay cleaning walkthrough: evidence, affected rows, and fixes

This notebook uses **pandas and PyArrow**, with no Spark or Java. It demonstrates the eight issues
identified in the replay audit, across all completed matches in `output/parsed`.

For each issue:
1. Read the explanation and the evidence cell.
2. Inspect the actual affected raw rows.
3. Run the separate fix cell and check the output and assertions.

Some raw values are normal replay representations that need transforming, rather than errors.
Kill/assist credit is not fully understood: the notebook demonstrates candidate rules and retains
ambiguous records for review instead of pretending they have been corrected.

**All changes are in memory.** Raw Parquet files and the shared contracts are not modified. This
walkthrough produces intermediate DataFrames, not a complete production cleaning job.

Install `python -m pip install -r notebooks/requirements.txt` from the repository root, select that
Python environment, then run cells in order. Setup detects the root from either the repository or
`notebooks/`. The final status table distinguishes verified steps from provisional results.

References: [raw contract](../docs/data-contract.md), [draft clean contract](../docs/clean-contract.md).

In [44]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "spark_jobs" / "replay_spec.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the notebook from the dota-pipeline project folder.")
PARSED = ROOT / "data" / "parsed"
FOCUS_MATCH_ID = None  # Set a completed match ID, or leave None to use the first.
NULL_HANDLE = 16777215
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
TABLE_NAMES = ["match", "players", "combatlog", "names", "entity_names", "gamerules",
               "heroes", "player_resource", "team_data", "wards", "item_entities"]

# Read physical files directly: partition columns may only exist in their folder names.
# Nullable pandas dtypes preserve absent integers and booleans instead of filling them with zero.
def read_table(name, match_id):
    partition = PARSED / name / f"match_id={match_id}"
    files = sorted(partition.glob("*.parquet"))
    if not files:
        if name == "wards":
            return None
        raise FileNotFoundError(f"No Parquet files: {partition}")
    df = pd.concat([pq.ParquetFile(f).read().to_pandas().convert_dtypes()
                    for f in files], ignore_index=True)
    if "match_id" in df and not df["match_id"].eq(match_id).all():
        raise ValueError(f"Embedded match_id disagrees with partition: {partition}")
    df["match_id"] = pd.Series(match_id, index=df.index, dtype="Int64")
    return df

# Null group keys are retained so audits do not silently discard missing values.
def grouped_counts(df, keys, label="rows"):
    return df.groupby(keys, dropna=False).size().rename(label).reset_index()

def duplicate_groups(df, keys):
    counts = grouped_counts(df, keys)
    return counts.loc[counts["rows"] > 1]

def last_rows(df, keys, time="tick", tie_breakers=()):
    # Highest tick first, lowest handle on a tied tick when requested.
    order = [*keys, time, *tie_breakers]
    return df.sort_values(order, ascending=[True] * len(keys) + [False] + [True] * len(tie_breakers),
                          kind="stable").drop_duplicates(keys)

def true_mask(series):
    return series.eq(True).fillna(False)

def false_mask(series):
    # Missing flags remain unknown; report exclusions rather than assuming False.
    return series.eq(False).fillna(False)

In [45]:
REPLAY_MATCH_IDS = sorted(int(p.stem) for p in (ROOT / "data").glob("*.dem") if p.stem.isdigit())
marker_root = PARSED / "_complete"
COMPLETE_MATCH_IDS = sorted(int(p.name.split("=", 1)[1]) for p in marker_root.glob("match_id=*")
    if p.name.split("=", 1)[1].isdigit() and any(p.glob("*.parquet")))
LOCAL_MATCH_IDS = sorted(set(REPLAY_MATCH_IDS) | set(COMPLETE_MATCH_IDS))
PENDING_MATCH_IDS = sorted(set(REPLAY_MATCH_IDS) - set(COMPLETE_MATCH_IDS))
print(f"Coverage: {len(COMPLETE_MATCH_IDS)}/{len(LOCAL_MATCH_IDS)} discovered matches fully parsed")
print("Ready:", COMPLETE_MATCH_IDS)
print("Unparsed local replays:", PENDING_MATCH_IDS)
print("Original .dem files still in data/:", len(REPLAY_MATCH_IDS))
if not COMPLETE_MATCH_IDS:
    raise RuntimeError(f"No completion markers found under {marker_root}. Check PARSED or parse replays first.")
MATCH_ID = FOCUS_MATCH_ID if FOCUS_MATCH_ID is not None else COMPLETE_MATCH_IDS[0]
if MATCH_ID not in COMPLETE_MATCH_IDS:
    raise ValueError(f"Focus match {MATCH_ID} is not completed.")
print("Detailed inspection match:", MATCH_ID)

batch = {}
coverage_rows = []
for name in TABLE_NAMES:
    parts = []
    for match_id in COMPLETE_MATCH_IDS:
        df = read_table(name, match_id)
        coverage_rows.append({"match_id": match_id, "table": name,
                              "rows": 0 if df is None else len(df), "partition_present": df is not None})
        if df is not None:
            parts.append(df)
    if parts:
        batch[name] = pd.concat(parts, ignore_index=True)
coverage = pd.DataFrame({"match_id": pd.Series(COMPLETE_MATCH_IDS, dtype="Int64")})
coverage_report = pd.DataFrame(coverage_rows)
display(coverage_report.pivot(index="match_id", columns="table", values="rows"))
print("Total loaded memory (MB):", round(sum(df.memory_usage(deep=True).sum()
                                          for df in batch.values()) / 1024**2, 1))

# Store one metric per match for a readable final summary.
metrics = {}
def report_counts(df, label):
    counts = grouped_counts(df, ["match_id"], label)
    result = coverage.merge(counts, on="match_id", how="left", validate="one_to_one")
    result[label] = result[label].fillna(0).astype("Int64")
    metrics[label] = result.set_index("match_id")[label]
    display(result)
    return result

def resolve_name(df, index_column, output_column):
    # Name indexes are local to each match. Null lookup keys cannot match.
    lookup = batch["names"][["match_id", "idx", "name"]].dropna(subset=["idx"]).rename(
        columns={"idx": index_column, "name": output_column})
    return df.merge(lookup, on=["match_id", index_column], how="left", validate="many_to_one")

Coverage: 10/10 discovered matches fully parsed
Ready: [9079737909, 9079737994, 9079738443, 9079738574, 9079740225, 9079740632, 9079741152, 9079743269, 9079745340, 9079746862]
Unparsed local replays: []
Original .dem files still in data/: 0
Detailed inspection match: 9079737909


table,combatlog,entity_names,gamerules,heroes,item_entities,match,names,player_resource,players,team_data,wards
match_id,,,,,,,,,,,
9079737909,21908,564,1508,14570,621,1,242,15080,10,15080,0
9079737994,31712,607,1348,14238,608,1,298,13480,10,13480,6
9079738443,14679,547,918,8154,342,1,218,9180,10,9180,10
9079738574,23895,605,1379,17525,1173,1,336,13790,10,13790,57
9079740225,6132,536,545,4320,185,1,155,5450,10,5450,5
9079740632,19802,579,1503,15356,439,1,271,15030,10,15030,6
9079741152,14863,581,1199,10740,443,1,291,12723,10,11990,17
9079743269,6613,555,761,24360,302,1,200,7610,10,7610,14
9079745340,10265,557,1079,10551,557,1,241,10790,10,10790,34


Total loaded memory (MB): 189.5


In [46]:
players_raw = batch["players"]
heroes_raw = batch["heroes"]
resource_raw = batch["player_resource"]
combat_raw = batch["combatlog"]
results = []

def record(issue, status, detail):
    results.append({"issue": issue, "status": status, "detail": detail})

def by_match(df, label="rows"):
    counts = grouped_counts(df, ["match_id"], label)
    out = coverage.merge(counts, on="match_id", how="left", validate="one_to_one")
    out[label] = out[label].fillna(0).astype("Int64")
    return out

def check_unique(df, key):
    assert not df[key].isna().any(axis=None), f"Null key: {key}"
    assert not df.duplicated(key).any(), f"Repeated key: {key}"

# Header names define the player roster within each match.
check_unique(players_raw, ["match_id", "slot"])
check_unique(players_raw, ["match_id", "hero_name"])

## Issue 1. Player identifiers use different numbering

**Why transform:** hero IDs such as 0, 2, ..., 18 cannot be joined directly to header slots 0-9.
Assist participant IDs are already in the 0-9 range. A universal divide-by-two rule is incorrect.

**Evidence cell:** compare raw hero identifiers with header slots, hero names, and teams. The
identity examples show why IDs differ; fractional assist slots demonstrate a wrong conversion.

In [47]:
id_examples = heroes_raw[["match_id", "class", "m_iPlayerID", "m_iTeamNum"]].drop_duplicates().copy()
id_examples["candidate_player_slot"] = id_examples["m_iPlayerID"] / 2
id_examples = id_examples.merge(players_raw[["match_id", "slot", "hero_name", "game_team"]],
    left_on=["match_id", "candidate_player_slot"], right_on=["match_id", "slot"],
    how="left", validate="many_to_one")
display(id_examples.sort_values(["match_id", "candidate_player_slot"]).head(12))
assist_id_examples = combat_raw["assist_players"].dropna().str.split(";").explode().str.strip()
assist_id_examples = pd.to_numeric(assist_id_examples[assist_id_examples.str.fullmatch(r"[0-9]+")]).drop_duplicates()
display(pd.DataFrame({"raw_assist_id": assist_id_examples,
    "wrong_if_divided_by_two": assist_id_examples / 2}).sort_values("raw_assist_id"))
print("Rows with raw hero IDs different from the required slot:",
      int(id_examples["m_iPlayerID"].ne(id_examples["candidate_player_slot"]).sum()))

,match_id,class,m_iPlayerID,m_iTeamNum,candidate_player_slot,slot,hero_name,game_team
3,9079737909,CDOTA_Unit_Hero_Juggernaut,0,2,0.0,0,npc_dota_hero_juggernaut,2
4,9079737909,CDOTA_Unit_Hero_Legion_Commander,2,2,1.0,1,npc_dota_hero_legion_commander,2
5,9079737909,CDOTA_Unit_Hero_Tidehunter,4,2,2.0,2,npc_dota_hero_tidehunter,2
6,9079737909,CDOTA_Unit_Hero_Axe,6,2,3.0,3,npc_dota_hero_axe,2
7,9079737909,CDOTA_Unit_Hero_Sniper,8,2,4.0,4,npc_dota_hero_sniper,2
8,9079737909,CDOTA_Unit_Hero_DrowRanger,10,3,5.0,5,npc_dota_hero_drow_ranger,3
9,9079737909,CDOTA_Unit_Hero_Sven,12,3,6.0,6,npc_dota_hero_sven,3
0,9079737909,CDOTA_Unit_Hero_Centaur,14,3,7.0,7,npc_dota_hero_centaur,3
1,9079737909,CDOTA_Unit_Hero_Huskar,16,3,8.0,8,npc_dota_hero_huskar,3
2,9079737909,CDOTA_Unit_Hero_Lina,18,3,9.0,9,npc_dota_hero_lina,3


,raw_assist_id,wrong_if_divided_by_two
3050,0,0.0
744,1,0.5
760,2,1.0
3070,3,1.5
838,4,2.0
771,5,2.5
203,6,3.0
863,7,3.5
785,8,4.0
762,9,4.5


Rows with raw hero IDs different from the required slot: 90


**Example: the same player has different identifiers in different tables**

In match `9079737909`:

```text
Hero: Lina
heroes.m_iPlayerID: 18
players.slot: 9
Assist participant ID for this player: 9
```

Joining hero ID `18` directly to header slot `9` would miss this player. For hero snapshots,
`18 / 2 = 9` gives the expected slot. Applying the same conversion to assist ID `9` would instead
produce `4.5`, which is not a player slot. Convert the identifier according to its source table.

**Fix:** convert validated hero IDs only, check the resulting team against the match roster, and
retain original rows that cannot be mapped. Do not divide assist IDs. The raw hero property is
kept in this intermediate frame so the conversion remains easy to inspect.

In [48]:
valid_hero_id = (heroes_raw["m_iPlayerID"].notna()
    & heroes_raw["m_iPlayerID"].mod(2).eq(0).fillna(False)
    & heroes_raw["m_iPlayerID"].between(0, 18).fillna(False))
hero_ids_for_review = heroes_raw.loc[~valid_hero_id].copy()
canonical_heroes = heroes_raw.loc[valid_hero_id].copy()
canonical_heroes["player_slot"] = (canonical_heroes["m_iPlayerID"] // 2).astype("Int64")
canonical_heroes = canonical_heroes.merge(players_raw[["match_id", "slot", "hero_name", "game_team"]].rename(
    columns={"slot": "player_slot"}), on=["match_id", "player_slot"], how="left", validate="many_to_one")
bad_team = canonical_heroes["hero_name"].isna() | canonical_heroes["m_iTeamNum"].ne(
    canonical_heroes["game_team"]).fillna(True)
hero_ids_for_review = pd.concat([hero_ids_for_review, canonical_heroes.loc[bad_team]], ignore_index=True)
canonical_heroes = canonical_heroes.loc[~bad_team].copy()
assert canonical_heroes["player_slot"].between(0, 9).all()
assert len(canonical_heroes) + len(hero_ids_for_review) == len(heroes_raw)
display(canonical_heroes[["match_id", "m_iPlayerID", "player_slot", "hero_name"]].drop_duplicates().head(12))
print("Unmapped hero rows retained for review:", len(hero_ids_for_review))
record("1. Hero player IDs", "Validated on loaded matches", f"{len(hero_ids_for_review)} hero rows retained for review")

,match_id,m_iPlayerID,player_slot,hero_name
0,9079737909,14,7,npc_dota_hero_centaur
1,9079737909,16,8,npc_dota_hero_huskar
2,9079737909,18,9,npc_dota_hero_lina
3,9079737909,0,0,npc_dota_hero_juggernaut
4,9079737909,2,1,npc_dota_hero_legion_commander
5,9079737909,4,2,npc_dota_hero_tidehunter
6,9079737909,6,3,npc_dota_hero_axe
7,9079737909,8,4,npc_dota_hero_sniper
8,9079737909,10,5,npc_dota_hero_drow_ranger
9,9079737909,12,6,npc_dota_hero_sven


Unmapped hero rows retained for review: 0


## Issue 2. Player-resource rows outside the roster

**Why filter:** only Radiant/Dire players with a valid team slot can become player statistics.
Rows for other teams or missing slots cannot be assigned to a player by guessing.

**Evidence cell:** show each invalid mapping and examples with ticks. The earlier audit found
733 rows at array index 10, team 1, with no team slot in match 9079741152. This is an extra array
entry, not evidence that 733 real players are missing.

In [49]:
resource = resource_raw.rename(columns={"m_vecPlayerData.m_iPlayerTeam": "team",
    "m_vecPlayerTeamData.m_iTeamSlot": "team_slot",
    "m_vecPlayerTeamData.m_nSelectedHeroID": "hero_id"}).copy()
valid_resource_mapping = resource["team"].isin([2, 3]) & resource["team_slot"].between(0, 4).fillna(False)
invalid_resource_rows = resource.loc[~valid_resource_mapping].copy()
display(by_match(invalid_resource_rows, "unmapped_rows"))
display(invalid_resource_rows[["match_id", "tick", "index", "team", "team_slot", "hero_id"]].head(10))
display(invalid_resource_rows.groupby(["match_id", "index", "team", "team_slot"], dropna=False).agg(
    rows=("tick", "size"), first_tick=("tick", "min"), last_tick=("tick", "max")).reset_index())

,match_id,unmapped_rows
0,9079737909,0
1,9079737994,0
2,9079738443,0
3,9079738574,0
4,9079740225,0
5,9079740632,0
6,9079741152,733
7,9079743269,0
8,9079745340,0
9,9079746862,0


,match_id,tick,index,team,team_slot,hero_id
76680,9079741152,14010,10,1,<NA>,<NA>
76691,9079741152,14040,10,1,<NA>,<NA>
76702,9079741152,14070,10,1,<NA>,<NA>
76713,9079741152,14100,10,1,<NA>,<NA>
76724,9079741152,14130,10,1,<NA>,<NA>
76735,9079741152,14160,10,1,<NA>,<NA>
76746,9079741152,14190,10,1,<NA>,<NA>
76757,9079741152,14220,10,1,<NA>,<NA>
76768,9079741152,14250,10,1,<NA>,<NA>
76779,9079741152,14280,10,1,<NA>,<NA>


,match_id,index,team,team_slot,rows,first_tick,last_tick
0,9079741152,10,1,<NA>,733,14010,35970


**Example: an array entry does not identify a Radiant or Dire player**

```text
Match: 9079741152
Tick: 14010
Player-resource array index: 10
Team: 1
Team slot: <NA>
Hero ID: <NA>
```

This entry appears in **733 snapshots**, from tick `14010` through `35970`. These are repeated
observations of one array entry, not 733 different players. The clean player roster uses teams
`2` and `3`, each with team slots `0` to `4`. Team `1` with no team slot cannot be mapped using
that rule. Exclude these rows from player statistics and retain them for inspection; do not
replace the missing team slot with `0` or guess which player they represent.

In [50]:
# Inpect match_id 9079741152
one_match = resource[resource["match_id"] == 9079741152]
display(one_match.groupby(["match_id", "index", "team", "team_slot"], dropna=False).agg(
    rows=("tick", "size"), first_tick=("tick", "min"), last_tick=("tick", "max")).reset_index())

,match_id,index,team,team_slot,rows,first_tick,last_tick
0,9079741152,0,2,0,1199,30,35970
1,9079741152,1,2,1,1199,30,35970
2,9079741152,2,2,2,1199,30,35970
3,9079741152,3,2,3,1199,30,35970
4,9079741152,4,2,4,1199,30,35970
5,9079741152,5,3,0,1199,30,35970
6,9079741152,6,3,1,1199,30,35970
7,9079741152,7,3,2,1199,30,35970
8,9079741152,8,3,3,1199,30,35970
9,9079741152,9,3,4,1199,30,35970


**Fix:** retain excluded rows separately, filter to teams 2/3 and slots 0-4, derive `player_slot`,
and check against the header roster. This follows the contract's instruction to skip non-playing
teams and avoids filling a missing team slot with zero.

In [51]:
resource_mapped = resource.loc[valid_resource_mapping].copy()
resource_mapped["player_slot"] = ((resource_mapped["team"] - 2) * 5 + resource_mapped["team_slot"]).astype("Int64")
roster = players_raw[["match_id", "slot", "game_team"]].rename(columns={"slot": "player_slot", "game_team": "header_team"})
resource_mapped = resource_mapped.merge(roster, on=["match_id", "player_slot"], how="left", validate="many_to_one")
assert resource_mapped["header_team"].notna().all()
assert resource_mapped["header_team"].eq(resource_mapped["team"]).all()
assert len(resource_mapped) + len(invalid_resource_rows) == len(resource_raw)
check_unique(resource_mapped, ["match_id", "player_slot", "tick"])
display(resource_mapped[["match_id", "tick", "index", "team", "team_slot", "player_slot"]].head(10))
print("Retained rows:", len(resource_mapped), "| Excluded rows kept for review:", len(invalid_resource_rows))
record("2. Non-roster resource rows", "Filtered with rows retained for review", f"{len(invalid_resource_rows)} rows excluded from player stats")

,match_id,tick,index,team,team_slot,player_slot
0,9079737909,30,0,2,0,0
1,9079737909,30,1,2,1,1
2,9079737909,30,2,2,2,2
3,9079737909,30,3,2,3,3
4,9079737909,30,4,2,4,4
5,9079737909,30,5,3,0,5
6,9079737909,30,6,3,1,6
7,9079737909,30,7,3,2,7
8,9079737909,30,8,3,3,8
9,9079737909,30,9,3,4,9


Retained rows: 113000 | Excluded rows kept for review: 733


## Issue 3. Hero selection is temporarily unset or changes

**Why transform:** a first snapshot can have hero ID 0; choosing the largest ID can select a
transient value such as 1000000. Neither is a reliable player-level hero assignment.

**Evidence cell:** show each player's observed hero-ID history and raw rows that disagree with
the final selection. A differing earlier value is historical state, not a value to overwrite in raw.

In [52]:
hero_history = resource_mapped.groupby(["match_id", "player_slot", "hero_id"], dropna=False).agg(
    rows=("tick", "size"), first_tick=("tick", "min"), last_tick=("tick", "max")).reset_index()
changed_players = hero_history.groupby(["match_id", "player_slot"], dropna=False).size().rename("observed_ids").reset_index()
changed_players = changed_players.loc[changed_players["observed_ids"] > 1]
display(hero_history.merge(changed_players[["match_id", "player_slot"]], on=["match_id", "player_slot"]).head(20))
final_selection = last_rows(resource_mapped, ["match_id", "player_slot"])[["match_id", "player_slot", "tick", "hero_id"]].rename(
    columns={"tick": "selection_tick", "hero_id": "final_hero_id"})
hero_discrepancy = resource_mapped.merge(final_selection, on=["match_id", "player_slot"], validate="many_to_one")
hero_discrepancy = hero_discrepancy.loc[hero_discrepancy["hero_id"].ne(hero_discrepancy["final_hero_id"]).fillna(True)]
display(hero_discrepancy[["match_id", "tick", "player_slot", "hero_id", "final_hero_id"]].head(12))

,match_id,player_slot,hero_id,rows,first_tick,last_tick
0,9079737909,1,0,2,30,60
1,9079737909,1,104,1506,90,45240
2,9079737909,2,0,5,30,150
3,9079737909,2,29,1503,180,45240
4,9079737909,4,0,2,30,60
5,9079737909,4,35,1506,90,45240
6,9079737909,5,0,2,30,60
7,9079737909,5,6,1506,90,45240
8,9079737909,6,0,4,30,120
9,9079737909,6,18,1503,180,45240


,match_id,tick,player_slot,hero_id,final_hero_id
1,9079737909,30,1,0,104
2,9079737909,30,2,0,29
4,9079737909,30,4,0,35
5,9079737909,30,5,0,6
6,9079737909,30,6,0,18
7,9079737909,30,7,0,96
11,9079737909,60,1,0,104
12,9079737909,60,2,0,29
14,9079737909,60,4,0,35
15,9079737909,60,5,0,6


**Example: the first and largest hero IDs are both misleading**

In match `9079737909`, the player at clean slot `6` has this selection history:

```text
Ticks 30-120: hero_id = 0
Tick 150:    hero_id = 1000000
Tick 180 onward, through the final snapshot: hero_id = 18
Header hero name: npc_dota_hero_sven
```

Selecting the first observed ID would return `0`; taking the maximum would return `1000000`.
Neither describes the final assignment in these samples. The fix below selects the final mapped
snapshot for the player-level table, which gives `18` here. Earlier rows stay unchanged in the raw
data. This is a demonstrated selection rule, not proof that a final snapshot covers every game mode.

**Fix for this walkthrough:** use the final mapped snapshot for the one-row-per-player table.
Convert an absent or nonpositive final selection to null, retaining it for review. Do not impose
an invented upper bound on hero IDs. Final selection works for these samples; agree this rule
with the contract owner before using it for matches with hero replacements or unusual modes.

In [53]:
selected = final_selection.copy()
selected["hero_id"] = selected["final_hero_id"].where(selected["final_hero_id"].gt(0)).astype("Int64")
hero_selection_for_review = selected.loc[selected["hero_id"].isna()].copy()
clean_players = players_raw.rename(columns={"slot": "player_slot", "game_team": "team"}).merge(
    selected[["match_id", "player_slot", "hero_id"]], on=["match_id", "player_slot"], how="left", validate="one_to_one")
clean_players["team_slot"] = (clean_players["player_slot"] % 5).astype("Int64")
check_unique(clean_players, ["match_id", "player_slot"])
assert len(clean_players) == len(players_raw)
display(clean_players.head(12))
print("Final selections needing review:", len(hero_selection_for_review))
record("3. Stable hero assignment", "Sample rule demonstrated", f"Final snapshot selected; {len(hero_selection_for_review)} missing final IDs")

,player_slot,hero_name,team,match_id,hero_id,team_slot
0,0,npc_dota_hero_juggernaut,2,9079737909,8,0
1,1,npc_dota_hero_legion_commander,2,9079737909,104,1
2,2,npc_dota_hero_tidehunter,2,9079737909,29,2
3,3,npc_dota_hero_axe,2,9079737909,2,3
4,4,npc_dota_hero_sniper,2,9079737909,35,4
5,5,npc_dota_hero_drow_ranger,3,9079737909,6,0
6,6,npc_dota_hero_sven,3,9079737909,18,1
7,7,npc_dota_hero_centaur,3,9079737909,96,2
8,8,npc_dota_hero_huskar,3,9079737909,59,3
9,9,npc_dota_hero_lina,3,9079737909,25,4


Final selections needing review: 0


## Issue 4. Replay ticks do not equal seconds on the game clock

**Why transform:** replays include pregame and postgame. A chart using `tick / 30` puts the horn
at a positive time rather than 0:00 and can overstate duration.

**Evidence cell:** show each match's first in-progress tick and actual early rows beside the
incorrect clock. Negative pregame values are valid and must not be clipped to zero.

In [54]:
game = batch["gamerules"][["match_id", "tick", "m_pGameRules.m_nGameState",
    "m_pGameRules.m_flGameStartTime", "m_pGameRules.m_flGameEndTime"]].rename(columns={
    "m_pGameRules.m_nGameState": "state", "m_pGameRules.m_flGameStartTime": "start_time",
    "m_pGameRules.m_flGameEndTime": "end_time"})
starts = game.loc[game["state"].eq(5)].groupby("match_id")["tick"].min().rename("game_start_tick").reset_index()
display(starts)
clock_evidence = game.merge(starts, on="match_id", how="left", validate="many_to_one")
clock_evidence["incorrect_tick_divided_by_30"] = clock_evidence["tick"] / 30.0
display(clock_evidence.loc[clock_evidence["tick"] < clock_evidence["game_start_tick"]].groupby("match_id").head(2).head(12))

,match_id,game_start_tick
0,9079737909,4260
1,9079737994,6150
2,9079738443,7290
3,9079738574,6240
4,9079740225,6120
5,9079740632,5580
6,9079741152,6480
7,9079743269,6030
8,9079745340,6060
9,9079746862,7110


,match_id,tick,state,start_time,end_time,game_start_tick,incorrect_tick_divided_by_30
0,9079737909,30,2,0.0,0.0,4260,1.0
1,9079737909,60,2,0.0,0.0,4260,2.0
1508,9079737994,30,2,0.0,0.0,6150,1.0
1509,9079737994,60,2,0.0,0.0,6150,2.0
2856,9079738443,30,2,0.0,0.0,7290,1.0
2857,9079738443,60,2,0.0,0.0,7290,2.0
3774,9079738574,30,2,0.0,0.0,6240,1.0
3775,9079738574,60,2,0.0,0.0,6240,2.0
5153,9079740225,30,2,0.0,0.0,6120,1.0
5154,9079740225,60,2,0.0,0.0,6120,2.0


**Example: replay time and the game clock have different zero points**

```text
Match: 9079737909
First sampled in-progress state (state 5): tick 4260
Raw snapshot: tick 30
Incorrect game time: 30 / 30 = 1 second
Adjusted game time: (30 - 4260) / 30 = -141 seconds
```

The row is approximately **2 minutes 21 seconds before the chosen game-start point**, rather
than one second into the game. At tick `4260`, the adjusted clock is zero. Negative values preserve
pregame activity; they are not invalid timestamps to delete. This convention uses a once-per-second
state snapshot, so it does not independently establish the exact instant when movement starts.

**Fix:** subtract each match's game-start tick, preserve negative values, and compute duration
from final start/end game clocks. Stop if these required timing values are unavailable, rather
than fabricating a duration from the replay end.

In [55]:
timing = coverage.merge(starts, on="match_id", how="left").merge(
    last_rows(game, ["match_id"])[["match_id", "start_time", "end_time"]], on="match_id", how="left", validate="one_to_one")
timing["duration_s"] = (timing["end_time"] - timing["start_time"]).where(timing["end_time"] > 0)
timing_for_review = timing.loc[timing["game_start_tick"].isna() | timing["duration_s"].isna()
                               | timing["duration_s"].le(0).fillna(False)]
assert timing_for_review.empty, "Timing missing/invalid: inspect timing_for_review before proceeding."
def add_game_time(df):
    out = df.merge(starts, on="match_id", how="left", validate="many_to_one")
    out["game_time_s"] = (out["tick"] - out["game_start_tick"]) / 30.0
    assert out["game_time_s"].notna().all()
    return out.drop(columns="game_start_tick")
clean_game_clock = add_game_time(game)
assert clean_game_clock.merge(starts, on="match_id").loc[lambda d: d["tick"].eq(d["game_start_tick"]), "game_time_s"].eq(0).all()
display(clean_game_clock.groupby("match_id").head(2).head(12))
display(timing)
record("4. Game clock", "Validated on loaded matches", "Horn mapped to zero; pregame negative; duration from game clocks")

,match_id,tick,state,start_time,end_time,game_time_s
0,9079737909,30,2,0.0,0.0,-141.0
1,9079737909,60,2,0.0,0.0,-140.0
1508,9079737994,30,2,0.0,0.0,-204.0
1509,9079737994,60,2,0.0,0.0,-203.0
2856,9079738443,30,2,0.0,0.0,-242.0
2857,9079738443,60,2,0.0,0.0,-241.0
3774,9079738574,30,2,0.0,0.0,-207.0
3775,9079738574,60,2,0.0,0.0,-206.0
5153,9079740225,30,2,0.0,0.0,-203.0
5154,9079740225,60,2,0.0,0.0,-202.0


,match_id,game_start_tick,start_time,end_time,duration_s
0,9079737909,4260,154.733337,1376.800049,1222.06665
1,9079737994,6150,223.233337,1222.366699,999.133362
2,9079738443,7290,260.5,791.833374,531.333374
3,9079738574,6240,240.300018,1164.93335,924.633301
4,9079740225,6120,220.200012,416.766693,196.566681
5,9079740632,5580,211.800018,1384.000122,1172.200073
6,9079741152,6480,232.133347,962.000122,729.86676
7,9079743269,6030,237.600006,653.833374,416.233368
8,9079745340,6060,226.233353,855.833374,629.600037
9,9079746862,7110,258.300018,824.666687,566.366699


In [56]:
# Add game time to raw combat events
pregame_activity = add_game_time(combat_raw)
pregame_activity = pregame_activity.loc[
    (pregame_activity["match_id"] == 9079737909)
    & (pregame_activity["game_time_s"] < 0)
].copy()

# Resolve attacker, target, and ability/item names
for index_col, name_col in [
    ("attacker_name_idx", "attacker"),
    ("target_name_idx", "target"),
    ("inflictor_name_idx", "source"),
]:
    pregame_activity = resolve_name(
        pregame_activity, index_col, name_col
    )

# For purchases, value contains the purchased item's name index
pregame_activity = resolve_name(
    pregame_activity, "value", "purchase_item"
)
pregame_activity.loc[
    pregame_activity["type"] != "DOTA_COMBATLOG_PURCHASE",
    "purchase_item",
] = pd.NA

pregame_activity = pregame_activity.sort_values(
    ["tick"], kind="stable"
)

# Count the types of activity recorded
display(
    pregame_activity.groupby("type")
    .size()
    .rename("event_count")
    .reset_index()
)

# Show actual activity
display(
    pregame_activity[
        [
            "game_time_s",
            "tick",
            "type",
            "attacker",
            "target",
            "source",
            "value",
            "purchase_item",
        ]
    ].head(50)
)

,type,event_count
0,DOTA_COMBATLOG_GAME_STATE,8
1,DOTA_COMBATLOG_GOLD,10
2,DOTA_COMBATLOG_MODIFIER_ADD,18
3,DOTA_COMBATLOG_MODIFIER_STACK_EVENT,12
4,DOTA_COMBATLOG_PURCHASE,6


,game_time_s,tick,type,attacker,target,source,value,purchase_item
0,-141.933333,2,DOTA_COMBATLOG_GAME_STATE,<NA>,<NA>,<NA>,2,<NA>
1,-141.933333,2,DOTA_COMBATLOG_GAME_STATE,<NA>,<NA>,<NA>,12,<NA>
2,-141.933333,2,DOTA_COMBATLOG_GAME_STATE,<NA>,<NA>,<NA>,9,<NA>
3,-136.733333,158,DOTA_COMBATLOG_GAME_STATE,<NA>,<NA>,<NA>,3,<NA>
4,-106.733333,1058,DOTA_COMBATLOG_GAME_STATE,<NA>,<NA>,<NA>,8,<NA>
5,-90.733333,1538,DOTA_COMBATLOG_GAME_STATE,<NA>,<NA>,<NA>,10,<NA>
6,-90.666667,1540,DOTA_COMBATLOG_GAME_STATE,<NA>,<NA>,<NA>,4,<NA>
7,-90.6,1542,DOTA_COMBATLOG_GOLD,<NA>,npc_dota_hero_juggernaut,<NA>,600,<NA>
8,-90.6,1542,DOTA_COMBATLOG_MODIFIER_ADD,npc_dota_hero_legion_commander,npc_dota_hero_legion_commander,modifier_legion_commander_outfight_them,0,<NA>
9,-90.6,1542,DOTA_COMBATLOG_GOLD,<NA>,npc_dota_hero_legion_commander,<NA>,600,<NA>


9	-90.6	1542	DOTA_COMBATLOG_GOLD	<NA>	npc_dota_hero_legion_commander	<NA>	600	<NA>
10	-90.6	1542	DOTA_COMBATLOG_GOLD	<NA>	npc_dota_hero_tidehunter	<NA>	600	<NA>

this attributes gold to each player at the start of the game

49	-0.133333	4256	DOTA_COMBATLOG_PURCHASE	<NA>	npc_dota_hero_sniper	<NA>	25	item_tango

this is the first item bought, just before the game starts. assume the game start timer starts when players can walk out of fountain. 

## Issue 5. Damage needs name resolution, a source key, and aggregation

**Why transform:** the clean table's grain is attacker/target/source/tick. Several raw hits can
share that grain. Dropping repeated keys loses damage. Many ordinary attacks have no source
index; the draft contract uses `dota_unknown` so the clean key is non-null.

**Evidence cell:** show the repeated-key counts, a complete example group, and missing-source rows.
Missing event-specific fields are not a reason to drop every sparse combat-log row.

In [57]:
damage = combat_raw.loc[combat_raw["type"].eq("DOTA_COMBATLOG_DAMAGE")
    & (true_mask(combat_raw["attacker_hero"]) | true_mask(combat_raw["target_hero"]))
    & false_mask(combat_raw["attacker_illusion"]) & false_mask(combat_raw["target_illusion"])].copy()
for index, label in [("attacker_name_idx", "attacker_name"), ("target_name_idx", "target_name"),
                     ("inflictor_name_idx", "inflictor_name")]:
    damage = resolve_name(damage, index, label)
damage_key = ["match_id", "tick", "attacker_name", "target_name", "inflictor_name"]
repeated_damage = duplicate_groups(damage, damage_key)
display(by_match(repeated_damage, "repeated_damage_groups"))
example_key = repeated_damage.sort_values("rows", ascending=False).head(1)[damage_key]
display(damage.merge(example_key, on=damage_key, how="inner")[damage_key + ["value"]])
print("not na inflictor name:")
display(
    damage.loc[
        ~damage["inflictor_name"].isna(),
        damage_key + ["value"],
    ]
)
missing_source = damage.loc[damage["inflictor_name_idx"].isna()]
display(by_match(missing_source, "missing_source_indexes"))
display(missing_source[["match_id", "tick", "attacker_name", "target_name", "inflictor_name_idx", "inflictor_name", "value"]].head(8))

,match_id,repeated_damage_groups
0,9079737909,256
1,9079737994,365
2,9079738443,743
3,9079738574,165
4,9079740225,135
5,9079740632,156
6,9079741152,160
7,9079743269,67
8,9079745340,90
9,9079746862,71


,match_id,tick,attacker_name,target_name,inflictor_name,value
0,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,9
1,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,9
2,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,18
3,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,18
4,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,9
5,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,9
6,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,19
7,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,19
8,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,9
9,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,<NA>,9


not na inflictor name:


,match_id,tick,attacker_name,target_name,inflictor_name,value
5,9079737909,5360,npc_dota_hero_sven,npc_dota_creep_goodguys_melee,sven_great_cleave,42
6,9079737909,5360,npc_dota_hero_sven,npc_dota_hero_tidehunter,sven_great_cleave,36
7,9079737909,5360,npc_dota_hero_sven,npc_dota_hero_legion_commander,sven_great_cleave,27
8,9079737909,5360,npc_dota_hero_sven,npc_dota_creep_goodguys_melee,sven_great_cleave,42
12,9079737909,5532,npc_dota_hero_sven,npc_dota_hero_tidehunter,sven_great_cleave,37
...,...,...,...,...,...,...
48947,9079746862,27088,npc_dota_hero_lina,npc_dota_creep_goodguys_ranged,lina_slow_burn,6
48952,9079746862,27114,npc_dota_hero_lina,npc_dota_creep_goodguys_melee,lina_slow_burn,46
48953,9079746862,27116,npc_dota_hero_lina,npc_dota_creep_goodguys_melee,lina_slow_burn,46
48957,9079746862,27144,npc_dota_hero_lina,npc_dota_creep_goodguys_melee,lina_slow_burn,13


,match_id,missing_source_indexes
0,9079737909,7505
1,9079737994,5630
2,9079738443,1489
3,9079738574,3383
4,9079740225,600
5,9079740632,4324
6,9079741152,2058
7,9079743269,1228
8,9079745340,1844
9,9079746862,1597


,match_id,tick,attacker_name,target_name,inflictor_name_idx,inflictor_name,value
0,9079737909,4954,npc_dota_hero_sniper,npc_dota_creep_badguys_melee,<NA>,<NA>,49
1,9079737909,5216,npc_dota_hero_sniper,npc_dota_creep_badguys_melee,<NA>,<NA>,63
2,9079737909,5256,npc_dota_hero_sniper,npc_dota_creep_badguys_melee,<NA>,<NA>,44
3,9079737909,5308,npc_dota_hero_juggernaut,npc_dota_creep_badguys_melee,<NA>,<NA>,47
4,9079737909,5354,npc_dota_hero_sniper,npc_dota_creep_badguys_melee,<NA>,<NA>,68
9,9079737909,5360,npc_dota_hero_sven,npc_dota_creep_goodguys_melee,<NA>,<NA>,18
10,9079737909,5398,npc_dota_hero_drow_ranger,npc_dota_creep_goodguys_melee,<NA>,<NA>,7
11,9079737909,5516,npc_dota_hero_sniper,npc_dota_creep_badguys_ranged,<NA>,<NA>,54


**Example: repeated damage keys can represent multiple hits**

The evidence above contains this group:

```text
Match: 9079738574
Tick: 23834
Attacker: npc_dota_broodmother_spiderling
Target: npc_dota_hero_phantom_lancer
Source: <NA>
Raw rows with this key: 16
Recorded damage values include: 9, 18, and 19
```

These rows share the clean table's key, but that does not make them unwanted copies. Dropping
all but one would lose recorded damage. The clean representation sums all 16 values and sets
`hits = 16`. A shared unit name can also represent multiple same-named entities, so this aggregation
is about the agreed name-based table grain, not identifying one physical attacker.

The absent source index becomes `dota_unknown` under the contract. It means the replay did not
supply a source index; it does not mean damage was zero. A non-null index that fails to resolve
is a separate lookup problem and is retained for review.

**Fix:** apply `dota_unknown` to absent source indexes only; retain unresolved non-null indexes
and invalid required values for review. Sum damage and count hits per clean key. Check both
uniqueness and conservation of the valid input's total damage and hit count.

In [58]:
damage_ready = damage.copy()
damage_ready.loc[damage_ready["inflictor_name_idx"].isna(), "inflictor_name"] = "dota_unknown"
invalid_damage = damage_ready[damage_key].isna().any(axis=1) | damage_ready["value"].isna() | damage_ready["value"].lt(0).fillna(False)
damage_for_review = damage_ready.loc[invalid_damage].copy()
damage_ready = damage_ready.loc[~invalid_damage].copy()
clean_damage = damage_ready.groupby(damage_key, dropna=False).agg(
    damage=("value", lambda s: s.sum(min_count=1)), hits=("value", "size")).reset_index()
for field, label in [("attacker_name", "attacker_slot"), ("target_name", "target_slot")]:
    clean_damage = clean_damage.merge(clean_players[["match_id", "hero_name", "player_slot"]].rename(
        columns={"hero_name": field, "player_slot": label}), on=["match_id", field], how="left", validate="many_to_one")
clean_damage = add_game_time(clean_damage)
check_unique(clean_damage, damage_key)
assert clean_damage["damage"].sum() == damage_ready["value"].sum()
assert clean_damage["hits"].sum() == len(damage_ready)
assert len(damage_ready) + len(damage_for_review) == len(damage)
display(clean_damage.sort_values("hits", ascending=False).head(8))
print("Input eligible rows:", len(damage), "| Clean groups:", len(clean_damage), "| Review rows:", len(damage_for_review))
record("5. Damage aggregation/source", "Validated on loaded matches", f"{len(clean_damage)} groups; damage and hit totals conserved; {len(damage_for_review)} review rows")

,match_id,tick,attacker_name,target_name,inflictor_name,damage,hits,attacker_slot,target_slot,game_time_s
26072,9079738574,23868,npc_dota_hero_oracle,npc_dota_broodmother_spiderling,oracle_fortunes_end,1200,16,0,<NA>,587.6
26067,9079738574,23798,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,dota_unknown,242,16,<NA>,2,585.266667
26070,9079738574,23834,npc_dota_broodmother_spiderling,npc_dota_hero_phantom_lancer,dota_unknown,221,16,<NA>,2,586.466667
25765,9079738574,21292,npc_dota_broodmother_spiderling,npc_dota_hero_arc_warden,dota_unknown,192,15,<NA>,1,501.733333
25757,9079738574,21256,npc_dota_broodmother_spiderling,npc_dota_hero_arc_warden,dota_unknown,190,15,<NA>,1,500.533333
11329,9079737994,18076,npc_dota_hero_bristleback,npc_dota_broodmother_spiderling,bristleback_quill_spray,570,12,4,<NA>,397.533333
11335,9079737994,18116,npc_dota_hero_sven,npc_dota_broodmother_spiderling,sven_great_cleave,564,12,3,<NA>,398.866667
26082,9079738574,23950,npc_dota_broodmother_spiderling,npc_dota_hero_oracle,dota_unknown,124,11,<NA>,0,590.333333


Input eligible rows: 48985 | Clean groups: 46043 | Review rows: 0


## Issue 6. Repeated purchases should become quantities

**Why transform:** buying the same item more than once at the same tick yields repeated clean
keys. Keeping one arbitrary row undercounts purchases. Names must first be resolved in the
correct match, then the buyer mapped to the roster.

**Evidence cell:** show repeated purchase groups and actual raw purchase rows from those groups.

In [59]:
purchases = resolve_name(resolve_name(combat_raw.loc[combat_raw["type"].eq("DOTA_COMBATLOG_PURCHASE")],
                                      "target_name_idx", "buyer"), "value", "item_name")
purchase_key_raw = ["match_id", "tick", "buyer", "item_name"]
repeated_purchases = duplicate_groups(purchases, purchase_key_raw)
display(by_match(repeated_purchases, "repeated_purchase_groups"))
display(purchases.merge(repeated_purchases[purchase_key_raw], on=purchase_key_raw, how="inner")[
    purchase_key_raw + ["target_name_idx", "value"]].head(15))

,match_id,repeated_purchase_groups
0,9079737909,0
1,9079737994,3
2,9079738443,1
3,9079738574,0
4,9079740225,0
5,9079740632,0
6,9079741152,0
7,9079743269,0
8,9079745340,0
9,9079746862,1


,match_id,tick,buyer,item_name,target_name_idx,value
0,9079737994,8914,npc_dota_hero_meepo,item_wraith_band,18,117
1,9079737994,8914,npc_dota_hero_meepo,item_wraith_band,18,117
2,9079737994,9644,npc_dota_hero_broodmother,item_wraith_band,13,117
3,9079737994,9644,npc_dota_hero_broodmother,item_wraith_band,13,117
4,9079737994,10952,npc_dota_hero_kunkka,item_bracer,21,136
5,9079737994,10952,npc_dota_hero_kunkka,item_bracer,21,136
6,9079738443,12640,npc_dota_hero_skywrath_mage,item_null_talisman,10,133
7,9079738443,12640,npc_dota_hero_skywrath_mage,item_null_talisman,10,133
8,9079746862,23426,npc_dota_hero_nyx_assassin,item_branches,1,95
9,9079746862,23426,npc_dota_hero_nyx_assassin,item_branches,1,95


**Example: two purchases become one row with quantity two**

```text
Match: 9079737994
Tick: 8914
Buyer: npc_dota_hero_meepo
Item: item_wraith_band
Raw purchase rows with this key: 2
```

The contract counts identical purchase events at one tick. Its output is one row with
`quantity = 2`. Simply dropping repeated keys would keep only one purchase and undercount the
recorded events. The fix checks that the sum of output quantities equals the valid input row count;
it does not independently prove how many items the player owned afterward.

**Fix:** resolve the buyer to a player slot, count each group into `quantity`, and retain rows
with unresolved keys for review. The sum of quantities must equal the number of valid input rows.

In [60]:
purchases_ready = purchases.merge(clean_players[["match_id", "hero_name", "player_slot"]].rename(
    columns={"hero_name": "buyer"}), on=["match_id", "buyer"], how="left", validate="many_to_one")
purchase_key = ["match_id", "tick", "player_slot", "item_name"]
invalid_purchase = purchases_ready[purchase_key].isna().any(axis=1)
purchases_for_review = purchases_ready.loc[invalid_purchase].copy()
purchases_ready = purchases_ready.loc[~invalid_purchase].copy()
clean_purchases = grouped_counts(purchases_ready, purchase_key, "quantity")
clean_purchases = add_game_time(clean_purchases)
check_unique(clean_purchases, purchase_key)
assert clean_purchases["quantity"].sum() == len(purchases_ready)
assert len(purchases_ready) + len(purchases_for_review) == len(purchases)
display(clean_purchases.loc[clean_purchases["quantity"] > 1])
print("Raw purchase rows:", len(purchases), "| Clean rows:", len(clean_purchases), "| Review rows:", len(purchases_for_review))
record("6. Purchase quantity", "Validated on loaded matches", f"Quantities conserve {len(purchases_ready)} valid purchase rows")

,match_id,tick,player_slot,item_name,quantity,game_time_s
225,9079737994,8914,9,item_wraith_band,2,92.133333
232,9079737994,9644,6,item_wraith_band,2,116.466667
239,9079737994,10952,8,item_bracer,2,160.066667
402,9079738443,12640,5,item_null_talisman,2,178.333333
1341,9079746862,23426,0,item_branches,2,543.866667


Raw purchase rows: 1365 | Clean rows: 1360 | Review rows: 0


## Issue 7. Assist lists include killers and use different ID semantics

**Why investigate:** splitting lists, dividing IDs by two, or counting every participant as an
assist produces incorrect scoreboard totals. The draft contract's divide-by-two assumption is
contradicted by odd IDs and by the player mappings already inspected.

**Evidence cell:** resolve real-hero deaths and show participant lists containing the direct killer,
plus raw participant counts compared with final player-resource assists. A missing killer slot
means the immediate attacker is not in the hero roster; it does not prove who got scoreboard credit.

In [61]:
deaths_named = resolve_name(resolve_name(combat_raw.loc[combat_raw["type"].eq("DOTA_COMBATLOG_DEATH")],
                                        "attacker_name_idx", "killer_name"), "target_name_idx", "victim_name")
deaths = deaths_named.loc[true_mask(deaths_named["target_hero"]) & false_mask(deaths_named["target_illusion"])
    & false_mask(deaths_named["attacker_illusion"])].merge(
    clean_players[["match_id", "hero_name", "player_slot"]].rename(columns={"hero_name": "victim_name", "player_slot": "victim_slot"}),
    on=["match_id", "victim_name"], how="inner", validate="many_to_one").merge(
    clean_players[["match_id", "hero_name", "player_slot"]].rename(columns={"hero_name": "killer_name", "player_slot": "killer_slot"}),
    on=["match_id", "killer_name"], how="left", validate="many_to_one")
assist_text = deaths[["match_id", "tick", "victim_slot", "killer_slot", "killer_name", "victim_name", "assist_players"]].copy()
assist_text["assist_token"] = assist_text["assist_players"].fillna("").str.split(";")
assist_text = assist_text.explode("assist_token", ignore_index=True)
assist_text["assist_token"] = assist_text["assist_token"].str.strip()
valid_assist_token = assist_text["assist_token"].str.fullmatch(r"[0-9]+").fillna(False)
participants = assist_text.loc[valid_assist_token].copy()
participants["assist_slot"] = pd.to_numeric(participants["assist_token"]).astype("Int64")
includes_killer = participants["assist_slot"].eq(participants["killer_slot"]).fillna(False)
display(by_match(participants.loc[includes_killer], "killer_listed_as_participant"))
display(participants.loc[includes_killer, ["match_id", "tick", "killer_name", "killer_slot", "victim_name", "assist_players"]].head(10))

last_stats = last_rows(resource_mapped, ["match_id", "player_slot"])[[
    "match_id", "player_slot", "m_vecPlayerTeamData.m_iKills", "m_vecPlayerTeamData.m_iDeaths", "m_vecPlayerTeamData.m_iAssists"]].rename(columns={
    "player_slot": "slot", "m_vecPlayerTeamData.m_iKills": "snapshot_kills",
    "m_vecPlayerTeamData.m_iDeaths": "snapshot_deaths", "m_vecPlayerTeamData.m_iAssists": "snapshot_assists"})
def add_event_count(base, events, slot_col, label):
    counts = grouped_counts(events.dropna(subset=[slot_col]), ["match_id", slot_col], label).rename(columns={slot_col: "slot"})
    out = base.merge(counts, on=["match_id", "slot"], how="left", validate="one_to_one")
    out[label] = out[label].fillna(0).astype("Int64")
    return out
assist_comparison = add_event_count(last_stats, participants, "assist_slot", "all_listed_participants")
display(assist_comparison[["match_id", "slot", "snapshot_assists", "all_listed_participants"]].head(15))

,match_id,killer_listed_as_participant
0,9079737909,26
1,9079737994,46
2,9079738443,5
3,9079738574,26
4,9079740225,8
5,9079740632,71
6,9079741152,22
7,9079743269,11
8,9079745340,10
9,9079746862,8


,match_id,tick,killer_name,killer_slot,victim_name,assist_players
0,9079737909,8224,npc_dota_hero_huskar,8,npc_dota_hero_sniper,8
1,9079737909,8450,npc_dota_hero_lina,9,npc_dota_hero_juggernaut,9;5
4,9079737909,8454,npc_dota_hero_tidehunter,2,npc_dota_hero_centaur,1;2
5,9079737909,8518,npc_dota_hero_legion_commander,1,npc_dota_hero_sven,1;2
7,9079737909,10674,npc_dota_hero_huskar,8,npc_dota_hero_sniper,8
9,9079737909,10686,npc_dota_hero_drow_ranger,5,npc_dota_hero_juggernaut,9;5
10,9079737909,10916,npc_dota_hero_lina,9,npc_dota_hero_axe,9;5
12,9079737909,10930,npc_dota_hero_legion_commander,1,npc_dota_hero_sven,1;2
14,9079737909,11004,npc_dota_hero_centaur,7,npc_dota_hero_tidehunter,7;6
16,9079737909,11388,npc_dota_hero_centaur,7,npc_dota_hero_legion_commander,7;6


,match_id,slot,snapshot_assists,all_listed_participants
0,9079737909,0,5,6
1,9079737909,1,4,7
2,9079737909,2,3,7
3,9079737909,3,4,6
4,9079737909,4,0,2
5,9079737909,5,5,6
6,9079737909,6,2,4
7,9079737909,7,2,4
8,9079737909,8,0,6
9,9079737909,9,1,6


**Example: a killer is also listed as a participant**

In match `9079737909`, at tick `8450`:

```text
Killer: Lina, player slot 9
Victim: Juggernaut
assist_players: "9;5"
```

Counting both IDs as assists would wrongly give Lina an assist on her own kill. Removing the
known direct killer, slot `9`, leaves slot `5` as an assist candidate. The IDs are used directly;
dividing `5` by two would create an invalid slot `2.5`.

This explains one source of inflation, but it does not settle every event's credit. The next cell
compares candidate totals with final scoreboard assists. A player-level disagreement identifies
an unresolved total; it does **not** prove that every displayed candidate event for that player
is individually wrong.

**Candidate fix, not yet final credit:** keep IDs directly as slots, validate against the roster,
exclude a known direct killer, then deduplicate the participant key. Retain malformed/invalid
participants. Compare again and explicitly keep remaining disagreements for review. Do not
replace the scoreboard with inferred numbers or call this a final `death_assists` table.

In [62]:
malformed_assists = assist_text.loc[assist_text["assist_token"].ne("") & ~valid_assist_token].copy()
participant_roster = clean_players[["match_id", "player_slot"]].rename(columns={"player_slot": "assist_slot"})
valid_participants = participants.merge(participant_roster.assign(in_roster=True),
    on=["match_id", "assist_slot"], how="left", validate="many_to_one")
invalid_participants = valid_participants.loc[valid_participants["in_roster"].isna()].copy()
valid_participants = valid_participants.loc[valid_participants["in_roster"].notna()].copy()
assist_key = ["match_id", "tick", "victim_slot", "assist_slot"]
assist_candidates = valid_participants.loc[valid_participants["killer_slot"].isna()
    | valid_participants["assist_slot"].ne(valid_participants["killer_slot"]).fillna(False)].drop_duplicates(assist_key).copy()
check_unique(assist_candidates, assist_key)
assert not assist_candidates["assist_slot"].eq(assist_candidates["killer_slot"]).fillna(False).any()
assist_comparison = add_event_count(assist_comparison, assist_candidates, "assist_slot", "candidate_assists")
assist_comparison["remaining_disagreement"] = assist_comparison["candidate_assists"].ne(assist_comparison["snapshot_assists"])
display(assist_comparison[["match_id", "slot", "snapshot_assists", "all_listed_participants", "candidate_assists", "remaining_disagreement"]].head(20))
assist_disagreements = assist_comparison.loc[assist_comparison["remaining_disagreement"].fillna(False)].copy()
display(by_match(assist_disagreements, "players_still_disagreeing"))
print("Malformed tokens:", len(malformed_assists), "| Unknown participant slots:", len(invalid_participants))
record("7. Assist participants", "Provisional: credit rule still needs agreement", f"{len(assist_disagreements)} players still differ after killer exclusion and key deduplication")

,match_id,slot,snapshot_assists,all_listed_participants,candidate_assists,remaining_disagreement
0,9079737909,0,5,6,5,False
1,9079737909,1,4,7,4,False
2,9079737909,2,3,7,3,False
3,9079737909,3,4,6,4,False
4,9079737909,4,0,2,2,True
5,9079737909,5,5,6,5,False
6,9079737909,6,2,4,2,False
7,9079737909,7,2,4,2,False
8,9079737909,8,0,6,0,False
9,9079737909,9,1,6,1,False


,match_id,players_still_disagreeing
0,9079737909,1
1,9079737994,6
2,9079738443,0
3,9079738574,4
4,9079740225,0
5,9079740632,4
6,9079741152,1
7,9079743269,1
8,9079745340,2
9,9079746862,1


Malformed tokens: 0 | Unknown participant slots: 0


In [68]:
# Players whose candidate assist counts differ from the scoreboard
disagreeing = assist_comparison.loc[
    assist_comparison["candidate_assists"]
    .ne(assist_comparison["snapshot_assists"])
    .fillna(False)
].copy()

disagreeing["difference"] = (
    disagreeing["candidate_assists"]
    - disagreeing["snapshot_assists"]
)

display(
    disagreeing[
        [
            "match_id",
            "slot",
            "snapshot_assists",
            "all_listed_participants",
            "candidate_assists",
            "difference",
        ]
    ].sort_values(["match_id", "slot"])
)

# Show the candidate assist events for those players
affected_players = disagreeing[["match_id", "slot"]].rename(
    columns={"slot": "assist_slot"}
)

display(
    assist_candidates.merge(
        affected_players,
        on=["match_id", "assist_slot"],
        how="inner",
        validate="many_to_one",
    )[
        [
            "match_id",
            "tick",
            "assist_slot",
            "killer_name",
            "killer_slot",
            "victim_name",
            "assist_players",
        ]
    ].sort_values(["match_id", "assist_slot", "tick"])
)

,match_id,slot,snapshot_assists,all_listed_participants,candidate_assists,difference
4,9079737909,4,0,2,2,2
13,9079737994,3,1,6,3,2
14,9079737994,4,2,7,3,1
16,9079737994,6,4,12,5,1
17,9079737994,7,13,18,14,1
18,9079737994,8,12,18,13,1
19,9079737994,9,9,24,10,1
30,9079738574,0,2,2,1,-1
31,9079738574,1,0,1,1,1
35,9079738574,5,10,13,7,-3


,match_id,tick,assist_slot,killer_name,killer_slot,victim_name,assist_players
0,9079737909,17682,4,npc_dota_goodguys_tower1_mid,<NA>,npc_dota_hero_huskar,4
1,9079737909,19898,4,npc_dota_goodguys_tower1_mid,<NA>,npc_dota_hero_huskar,4
32,9079737994,20562,3,npc_dota_neutral_centaur_outrunner,<NA>,npc_dota_hero_grimstroke,3
35,9079737994,21912,3,npc_dota_neutral_big_thunder_lizard,<NA>,npc_dota_hero_shadow_shaman,3
40,9079737994,25880,3,npc_dota_hero_bristleback,4,npc_dota_hero_shadow_shaman,4;3
...,...,...,...,...,...,...,...
105,9079743269,17222,2,npc_dota_hero_dawnbreaker,4,npc_dota_hero_monkey_king,2;4
106,9079745340,18080,9,npc_dota_hero_spirit_breaker,8,npc_dota_hero_slark,9;8
107,9079745340,21774,9,npc_dota_hero_spirit_breaker,8,npc_dota_hero_slark,9;8
108,9079745340,21952,9,npc_dota_hero_spirit_breaker,8,npc_dota_hero_vengefulspirit,9;8


In [69]:
assist_difference = assist_comparison[
    ["match_id", "slot", "snapshot_assists", "candidate_assists"]
].copy()

assist_difference = assist_difference.rename(
    columns={"snapshot_assists": "scoreboard_assists"}
)

assist_difference["difference"] = (
    assist_difference["candidate_assists"]
    - assist_difference["scoreboard_assists"]
)

display(
    assist_difference.loc[
        assist_difference["difference"].ne(0).fillna(False)
    ].sort_values(["match_id", "slot"])
)

,match_id,slot,scoreboard_assists,candidate_assists,difference
4,9079737909,4,0,2,2
13,9079737994,3,1,3,2
14,9079737994,4,2,3,1
16,9079737994,6,4,5,1
17,9079737994,7,13,14,1
18,9079737994,8,12,13,1
19,9079737994,9,9,10,1
30,9079738574,0,2,1,-1
31,9079738574,1,0,1,1
35,9079738574,5,10,7,-3


## Issue 8. Conflicting deaths and kill-credit differences

**Why investigate:** some same-key death rows have different attackers, sources, or flags. They
are not exact copies. The direct attacker may be a tower or summon while a hero gets scoreboard
credit. Arbitrarily dropping a row or assigning every non-player kill to a participant is unsafe.

**Evidence cell:** display all rows for repeated death keys, including the differing fields, then
show players whose direct event counts disagree with final kills/deaths.

In [63]:
death_key = ["match_id", "tick", "victim_slot"]
repeated_death_groups = duplicate_groups(deaths, death_key)
conflicting_deaths = deaths.merge(repeated_death_groups[death_key], on=death_key, how="inner", validate="many_to_one")
display(by_match(repeated_death_groups, "repeated_death_key_groups"))
print("Exact duplicate full death records:", int(deaths.duplicated().sum()))
evidence_columns = death_key + ["killer_name", "killer_slot", "victim_name", "inflictor_name_idx", "assist_players",
    "target_self", "damage_category", "will_reincarnate"]
display(conflicting_deaths[evidence_columns].sort_values(death_key).head(20))
credit_comparison = add_event_count(last_stats, deaths, "killer_slot", "direct_event_kills")
credit_comparison = add_event_count(credit_comparison, deaths, "victim_slot", "raw_event_deaths")
credit_mismatch = credit_comparison["direct_event_kills"].ne(credit_comparison["snapshot_kills"]) | credit_comparison["raw_event_deaths"].ne(credit_comparison["snapshot_deaths"])
display(credit_comparison.loc[credit_mismatch.fillna(False)])
display(deaths.loc[deaths["killer_slot"].isna(),
    ["match_id", "tick", "killer_name", "victim_name", "assist_players"]].head(15))

,match_id,repeated_death_key_groups
0,9079737909,0
1,9079737994,2
2,9079738443,0
3,9079738574,0
4,9079740225,0
5,9079740632,0
6,9079741152,0
7,9079743269,0
8,9079745340,0
9,9079746862,0


Exact duplicate full death records: 0


,match_id,tick,victim_slot,killer_name,killer_slot,victim_name,inflictor_name_idx,assist_players,target_self,damage_category,will_reincarnate
0,9079737994,25036,9,npc_dota_hero_meepo,9,npc_dota_hero_meepo,235,<NA>,True,694,False
1,9079737994,25036,9,npc_dota_hero_sven,3,npc_dota_hero_meepo,<NA>,4;3,False,1,False
2,9079737994,33150,9,npc_dota_hero_meepo,9,npc_dota_hero_meepo,235,4,True,694,False
3,9079737994,33150,9,npc_dota_hero_meepo,9,npc_dota_hero_meepo,235,<NA>,False,694,False
4,9079737994,33150,9,npc_dota_hero_bristleback,4,npc_dota_hero_meepo,50,4,False,0,False


,match_id,slot,snapshot_kills,snapshot_deaths,snapshot_assists,direct_event_kills,raw_event_deaths
4,9079737909,4,2,8,0,0,8
16,9079737994,6,8,1,4,7,1
17,9079737994,7,5,1,13,4,1
18,9079737994,8,6,4,12,5,4
19,9079737994,9,16,2,9,17,5
31,9079738574,1,1,5,0,0,5
36,9079738574,6,7,1,2,5,1
51,9079740632,1,37,0,7,36,0
55,9079740632,5,1,17,0,0,17
59,9079740632,9,1,15,0,0,15


,match_id,tick,killer_name,victim_name,assist_players
10,9079737909,12918,npc_dota_creep_goodguys_melee,npc_dota_hero_drow_ranger,0;3
12,9079737909,13192,npc_dota_creep_goodguys_melee,npc_dota_hero_lina,0;3
23,9079737909,17682,npc_dota_goodguys_tower1_mid,npc_dota_hero_huskar,4
25,9079737909,17998,npc_dota_goodguys_tower1_bot,npc_dota_hero_lina,3;0
28,9079737909,19546,npc_dota_creep_goodguys_melee,npc_dota_hero_lina,<NA>
29,9079737909,19898,npc_dota_goodguys_tower1_mid,npc_dota_hero_huskar,4
32,9079737909,30388,npc_dota_neutral_dark_troll_warlord,npc_dota_hero_lina,<NA>
33,9079737909,30648,npc_dota_badguys_tower2_top,npc_dota_hero_sniper,<NA>
34,9079737909,34306,npc_dota_creep_badguys_ranged,npc_dota_hero_sniper,<NA>
35,9079737909,40458,npc_dota_goodguys_tower3_bot,npc_dota_hero_sven,<NA>


**Example: same death key, different records**

```text
Match: 9079737994
Tick: 25036
Victim: Meepo, player slot 9
Record A attacker: Meepo, player slot 9; target_self = True
Record B attacker: Sven, player slot 3; target_self = False
```

Both rows map to the same clean death key `(match_id, tick, victim_slot)`, but their attacker and
other values differ. They are not exact full-record duplicates. Keeping either one arbitrarily
could misrepresent the death or kill credit, so the fix below holds the whole conflicting key
for review instead of inventing a winner.

There is a separate credit discrepancy in match `9079737909`:

```text
Player: Sniper, slot 4
Final scoreboard kills: 2
Kills from direct attacker-name mapping: 0
Two tower-caused Huskar deaths list participant "4"
```

This suggests immediate attacker and scoreboard credit need separate treatment. It does not prove
a universal rule for tower kills. Unique-key previews remain incomplete while these semantics
are unresolved; no code below forces counts to equal the scoreboard.

**Safe handling:** remove only exact full-record duplicates, and quarantine every remaining
conflicting key instead of choosing an arbitrary death. Separate uniquely represented deaths
with unresolved non-player kill credit for attribution review as well. Produce a structural
preview with unique keys; it is intentionally incomplete and is not ready to load downstream.
Keep related assist candidates on hold for conflicting deaths, too.

This resolves the uniqueness problem conservatively, not scoreboard attribution. The original
raw rows, snapshot totals, conflicts, and unresolved credit evidence remain available for Chris.

In [64]:
exact_unique_deaths = deaths.drop_duplicates().copy()
conflict_keys = duplicate_groups(exact_unique_deaths, death_key)[death_key]
marked = exact_unique_deaths.merge(conflict_keys.assign(conflict=True), on=death_key, how="left", validate="many_to_one")
deaths_for_review = marked.loc[marked["conflict"].eq(True).fillna(False)].drop(columns="conflict").copy()
unambiguous_deaths = marked.loc[marked["conflict"].isna()].drop(columns="conflict").copy()
check_unique(unambiguous_deaths, death_key)
assert len(exact_unique_deaths) == len(unambiguous_deaths) + len(deaths_for_review)
kill_credit_for_review = unambiguous_deaths.loc[unambiguous_deaths["killer_slot"].isna()].copy()
death_preview = add_game_time(unambiguous_deaths[["match_id", "tick", "victim_slot", "killer_slot", "killer_name"]])
assist_marked = assist_candidates.merge(conflict_keys.assign(conflict=True), on=death_key, how="left", validate="many_to_one")
assist_conflicts_for_review = assist_marked.loc[assist_marked["conflict"].eq(True).fillna(False)].copy()
assist_preview = assist_marked.loc[assist_marked["conflict"].isna()].drop(columns="conflict").copy()
check_unique(assist_preview, assist_key)
display(death_preview.head(10))
print("Unique-key death preview rows:", len(death_preview))
print("Conflicting death rows retained:", len(deaths_for_review))
print("Non-player attacker rows requiring kill-credit review:", len(kill_credit_for_review))
print("Assist candidate rows held for conflicting deaths:", len(assist_conflicts_for_review))
post_handling = add_event_count(last_stats, unambiguous_deaths, "victim_slot", "preview_deaths")
display(post_handling.loc[post_handling["preview_deaths"].ne(post_handling["snapshot_deaths"]).fillna(False),
    ["match_id", "slot", "snapshot_deaths", "preview_deaths"]])
record("8. Death conflicts and attribution", "Incomplete: conflicts retained for review", f"{len(deaths_for_review)} conflicting rows; {len(kill_credit_for_review)} non-player-attacker rows; no invented credit")

,match_id,tick,victim_slot,killer_slot,killer_name,game_time_s
0,9079737909,8224,4,8,npc_dota_hero_huskar,132.133333
1,9079737909,8450,0,9,npc_dota_hero_lina,139.666667
2,9079737909,8454,7,2,npc_dota_hero_tidehunter,139.8
3,9079737909,8518,6,1,npc_dota_hero_legion_commander,141.933333
4,9079737909,10674,4,8,npc_dota_hero_huskar,213.8
5,9079737909,10686,0,5,npc_dota_hero_drow_ranger,214.2
6,9079737909,10916,3,9,npc_dota_hero_lina,221.866667
7,9079737909,10930,6,1,npc_dota_hero_legion_commander,222.333333
8,9079737909,11004,2,7,npc_dota_hero_centaur,224.8
9,9079737909,11388,1,7,npc_dota_hero_centaur,237.6


Unique-key death preview rows: 275
Conflicting death rows retained: 5
Non-player attacker rows requiring kill-credit review: 44
Assist candidate rows held for conflicting deaths: 2


,match_id,slot,snapshot_deaths,preview_deaths
19,9079737994,9,2,0


## What has been fixed, and what remains open

The validated assertions check roster mapping, key uniqueness, row accounting, game-clock
conversion, and conservation of damage/hits/purchase quantities. Candidate rules are explicitly
labelled, and unresolved rows remain in named review DataFrames.

`canonical_heroes`, `resource_mapped`, `clean_players`, `clean_damage`, and `clean_purchases` are
in-memory transformations to study before translating into the production Spark clean job.
`assist_candidates`, `assist_preview`, and `death_preview` remain provisional; the presence of
unique keys alone does not validate their credit semantics or completeness.

In [65]:
status = pd.DataFrame(results)
with pd.option_context("display.max_colwidth", None):
    display(status)
review_counts = pd.DataFrame([
    ("hero_ids_for_review", len(hero_ids_for_review)),
    ("invalid_resource_rows", len(invalid_resource_rows)),
    ("hero_selection_for_review", len(hero_selection_for_review)),
    ("timing_for_review", len(timing_for_review)),
    ("damage_for_review", len(damage_for_review)),
    ("purchases_for_review", len(purchases_for_review)),
    ("assist_disagreements", len(assist_disagreements)),
    ("deaths_for_review", len(deaths_for_review)),
    ("kill_credit_for_review", len(kill_credit_for_review)),
], columns=["review_dataframe", "rows"])
display(review_counts)

,issue,status,detail
0,1. Hero player IDs,Validated on loaded matches,0 hero rows retained for review
1,2. Non-roster resource rows,Filtered with rows retained for review,733 rows excluded from player stats
2,3. Stable hero assignment,Sample rule demonstrated,Final snapshot selected; 0 missing final IDs
3,4. Game clock,Validated on loaded matches,Horn mapped to zero; pregame negative; duration from game clocks
4,5. Damage aggregation/source,Validated on loaded matches,46043 groups; damage and hit totals conserved; 0 review rows
5,6. Purchase quantity,Validated on loaded matches,Quantities conserve 1365 valid purchase rows
6,7. Assist participants,Provisional: credit rule still needs agreement,20 players still differ after killer exclusion and key deduplication
7,8. Death conflicts and attribution,Incomplete: conflicts retained for review,5 conflicting rows; 44 non-player-attacker rows; no invented credit


,review_dataframe,rows
0,hero_ids_for_review,0
1,invalid_resource_rows,733
2,hero_selection_for_review,0
3,timing_for_review,0
4,damage_for_review,0
5,purchases_for_review,0
6,assist_disagreements,20
7,deaths_for_review,5
8,kill_credit_for_review,44
